- 3a Stochastic pulse distribution
- 3b Noise floor
- 3c Trigger and acquisition window
- 3d Charge integration
- 3e Neutron/gamma channels (3D plot)

In [ ]:
# Imports
# import pickle
from typing import Callable, Literal
from random import sample
from statistics import mean
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mpl_toolkits.mplot3d.art3d as art3d
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
from scipy.optimize import curve_fit
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn, get_df_col
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
# Functions
pileup_flag = 0x8000


def is_pileup_flag(flags: int) -> bool:
    return flags & pileup_flag != 0

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 30,
    max_adc: int = 16367,
    baseline_offset: float = 0.10
) -> pd.DataFrame:
    # offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    # baselines = signals_np[:, :baseline_idx_range].mean(axis=1).reshape(-1, 1)
    # signals_np = -signals_np + baselines + offset
    signals_np = -signals_np + max_adc
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def textbox(text, x, y, width, height, facecolor, textcolor, ax, text_x_offset=0.5, text_y_offset=0.5, ha="center"):
    rect_params = {
        "linewidth": 0,
        # "ec": "black",
        "fc": facecolor
    }
    text_params = {
        "ha": ha,
        "va": "center",
        "fontsize": fontsize,
        "color": textcolor
    }
    rect = mpl.patches.Rectangle((x, y), width, height, **rect_params)
    ax.add_patch(rect)
    # poly = mpl.patches.Polygon(
    ax.annotate(
        # (t_2 + t_3) / 2,
        text,
        (text_x_offset, text_y_offset),
        xycoords=rect,
        **text_params
    )

In [ ]:
# def get_gamma_neutron_color(
#     is_gamma: bool,
#     is_neutron: bool,
#     is_uncertain: bool,
#     psd: float,
#     gamma_color: ColorAlphaTuple,
#     neutron_color: ColorAlphaTuple
# ) -> ColorAlphaTuple | None:
#     if is_uncertain:
#         psd_unit_interval = to_unit_interval(psd, 0.2, 0.3)
#         return calculate_gradient_color(psd_unit_interval, gamma_color, neutron_color)
#     elif is_gamma:
#         return gamma_color
#     elif is_neutron:
#         return neutron_color
#     else:
#         return None


# def text3d(ax, xyz, s, zdir="z", size=None, angle=0, usetex=False, **kwargs):
#     """
#     Plots the string *s* on the Axes *ax*, with position *xyz*, size *size*,
#     and rotation angle *angle*. *zdir* gives the axis which is to be treated as
#     the third dimension. *usetex* is a boolean indicating whether the string
#     should be run through a LaTeX subprocess or not.  Any additional keyword
#     arguments are forwarded to `.transform_path`.

#     Originally created by matplotlib development team.
#     See https://matplotlib.org/stable/gallery/mplot3d/pathpatch3d.html

#     Note: zdir affects the interpretation of xyz.
#     """
#     x, y, z = xyz
#     if zdir == "y":
#         xy1, z1 = (x, z), y
#     elif zdir == "x":
#         xy1, z1 = (y, z), x
#     else:
#         xy1, z1 = (x, y), z

#     text_path = mpl.text.TextPath((0, 0), s, size=size, usetex=usetex)
#     trans = mpl.transforms.Affine2D().rotate(angle).translate(xy1[0], xy1[1])

#     p1 = mpl.patches.PathPatch(trans.transform_path(text_path), **kwargs)
#     ax.add_patch(p1)
#     art3d.pathpatch_2d_to_3d(p1, z=z1, zdir=zdir)


def get_bbox_center(bbox: mpl.transforms.BboxBase) -> tuple[float, float]:
    return (bbox.xmin + bbox.width / 2, bbox.ymin + bbox.height / 2)


def get_translation_to(
    pos_from: tuple[float, float],
    pos_to: tuple[float, float]
) -> tuple[float, float]:
    x_from, y_from = pos_from
    x_to, y_to = pos_to
    return (x_to - x_from, y_to - y_from)


def make_text_path(
    text: str | list[str],
    # position: tuple[float, float],
    # rotation: float,
    # scaling: tuple[float, float],
    linespacing: float,
    font_properties=None,
    usetex=False
) -> mpl.text.TextPath:
    # scaling_x, scaling_y = scaling
    # pos_x, pos_y = position
    paths = []
    
    if isinstance(text, str):
        text = [text]
    
    for i, line in enumerate(text):
        text_path = mpl.text.TextPath(
            (0, 0), line,
            prop=font_properties,
            size=1,
            usetex=usetex
        )
        bbox = text_path.get_extents()
        line_width = bbox.width
        from_x = bbox.xmin
        from_y = bbox.xmax
        # center first line on x=0, anchor at y=0, next lines below
        to_x = -line_width / 2
        to_y = -i * linespacing
        x = to_x - from_x
        y = to_y - from_y
        
        line_transform = mpl.transforms.Affine2D().translate(x, y)
        transformed_path = line_transform.transform_path(text_path)
        paths.append(transformed_path)
    
    text_path = mpl.path.Path.make_compound_path(*paths)
    return text_path


def make_bounding_box_for_text_path(
    text_path: mpl.text.TextPath,
    padding: float,
    rounding_size: float,
    zorder: int,
    mutation_aspect: float = 1,
    **bbox_params
) -> mpl.patches.FancyBboxPatch:
    text_bbox = text_path.get_extents()
    # get anchor corner, width, height
    # make FancyBboxPatch
    anchor = text_bbox.min
    boxstyle = f"round, pad={padding}, rounding_size={rounding_size}"
    return mpl.patches.FancyBboxPatch(
        anchor,
        text_bbox.width,
        text_bbox.height,
        boxstyle=boxstyle,
        mutation_aspect=mutation_aspect,
        **bbox_params
    )


def convert_text_path_to_patch(
    text_path: mpl.text.TextPath,
    zorder: int
) -> mpl.patches.PathPatch:
    return mpl.patches.PathPatch(text_path, ec="none", fc="k", zorder=zorder)


def patch_to_3d_plot_wall(
    patch: mpl.patches.Patch,
    ax: mpl.axes.Axes,
    zdir: str,
    z: float = 0,
):
    ax.add_patch(patch)
    art3d.pathpatch_2d_to_3d(patch, z=z, zdir=zdir)


def get_center_match_transform(
    box_from: mpl.transforms.Bbox,
    box_to: mpl.transforms.Bbox
) -> tuple[float, float]:
    from_c_x = (box_from.x0 + box_from.x1) / 2
    from_c_y = (box_from.y0 + box_from.y1) / 2
    to_c_x = (box_to.x0 + box_to.x1) / 2
    to_c_y = (box_to.y0 + box_to.y1) / 2
    return (to_c_x - from_c_x, to_c_y - from_c_y)

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["test_noise_floor_26.04.24", "TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
unconv_folder = Path("Q:/Neutron Data/1-Unconverted_Data")
long_window_dataset_name = "TB-long_window"
exp_folder = unconv_folder / long_window_dataset_name
# [x for x in exp_folder.iterdir()]

In [ ]:
raw_folder = exp_folder / "RAW"
# print([x for x in raw_folder.iterdir() if x.suffix.lower() == ".csv"])
data_file = raw_folder / "SDataR_TB-long_window.CSV"
# data_file.is_file()

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    if exp_id == "TB-26":
        exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
headers = []
# samples_str = []
samples_lines = []
# flags_idx = 0
samples_start_idx = 0
scan_start = 1
scan_len = 1000
scan_end = scan_start + scan_len
last_idx = 0

with open(data_file) as fp:
    for i, line in enumerate(fp):
        if i == 0:
            headers = line.strip().split(";")
            # flags_idx = headers.index("FLAGS")
            samples_start_idx = headers.index("SAMPLES")
        elif scan_start <= i < scan_end:
            possible_line = line.strip().split(";")[samples_start_idx:]
            # flags = int(possible_line[flags_idx], base=16)
            # if is_pileup_flag(flags):
            #     print(f"Pileup found at index {i}")
            #     samples_str = possible_line[samples_start_idx:]
            #     break
            samples_lines.append(possible_line)
            last_idx = i
        elif i >= scan_end:
            # print("No pileup found")
            break
# samples_str = samples_str[len(headers)-1:]
# print(headers)
# print(samples_str[:15])
# print(len(samples_str))
# print(last_idx)
samples = np.array(samples_lines, dtype=int)
print(samples.shape)
exp_data = {"samples": samples}
experiment_neutron_data[long_window_dataset_name] = exp_data

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = load.calculate_timetag_hours(unclassified_df)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Figure 3b Processing

In [ ]:
exp_id = "test_noise_floor_26.04.24"

In [ ]:
exp_data = experiment_neutron_data[exp_id]
signals_df = exp_data["signals_df"]
signals_df = signals_df.astype("int32")
signals_np = signals_df.to_numpy()
baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
signals_np = -signals_np + baselines
signals_df.columns = signals_df.columns.map(int)
signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
exp_data["signals_df"] = signals_df

In [ ]:
exp_data = experiment_neutron_data[exp_id]
signals_df = exp_data["signals_df"]
histo_data = signals_df.iloc[:, :40]
histo_times = np.tile(
    histo_data.columns.map(lambda x: x * 2),
    histo_data.shape[0]
)
histo_es = np.ravel(histo_data.to_numpy(), order="F")
# print(histo_es.min())
# print(histo_es.max())

time_bins = np.arange(0, 82, 2)
e_min = -500
e_max = 750
e_bins = np.linspace(e_min, e_max, int((e_max - e_min) / 10)+1)
histo_results = np.histogram2d(histo_times, histo_es, bins=(time_bins, e_bins))
histo_counts, histo_time_bins, histo_e_bins = histo_results
collapsed_histo_energy = histo_counts.sum(axis=0)
exp_data["histo_results"] = {
    "counts": histo_counts,
    "time_bins": histo_time_bins,
    "e_bins": histo_e_bins,
    "collapsed_histo_energy": collapsed_histo_energy
}

In [ ]:
exp_data = experiment_neutron_data[exp_id]
histo_results = exp_data["histo_results"]
collapsed_histo = histo_results["collapsed_histo_energy"]
e_bins = histo_results["e_bins"]
e_mids = (e_bins[1:] + e_bins[:-1]) / 2

fit_params, _ = curve_fit(proc.gaussian, e_mids, collapsed_histo)
print(fit_params)
histo_results["fit_params"] = fit_params

### Figure 3c/d Processing

#### Neutron Classification

In [ ]:
exp_id = "TB-26"

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
Z, xe, ye = proc.get_psd_energy_histogram(
    psd_report,
    calibrated_energy_column,
    energy_width=energy_width
)
exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
exp_data = experiment_neutron_data[exp_id]
stop_here = False

exp_data = experiment_neutron_data[exp_id]
Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

# # Default
# default_bounds: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
# )

# bounds_a: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
# )

# bounds_b: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
# )

# # Ranged Example
# bounds = [
#     ((0, 60), bounds_a),
# ]

df, df_err = proc.scan_histogram_slices(
    Z,
    xe,
    ye,
    fit_style="peak_finder",
    # default_bounds,
    # bounds=bounds,
    start_idx=start_scan_idx,
    end_idx=end_scan_idx
)
df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

if bad_slice_indexes is not None:
    exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
    exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
    stop_here = True
else:
    # exp_data['fom_results'] = df
    exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
exp_data = experiment_neutron_data[exp_id]
if ExperimentDataKey.FOM_RESULTS not in exp_data:
    print(f"No good fit data on Experiment {exp_id}")
else:
    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]
    
    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()
    
    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
borders = exp_data[ExperimentDataKey.BORDERS]

psd_report = proc.classify(
    psd_report,
    calibrated_energy_column,
    borders,
    DetectorDataframeColumn.NEW_N_CLASS
)

exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

#### Pulse Selection

In [ ]:
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value

gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
neutrons_only = psd_report.query(n_class_col_name).copy()
exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
exp_data = experiment_neutron_data[exp_id]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
signals_df = exp_data["signals_df"]

neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
gamma_signals = signals_df.loc[gamma_only.index].astype("int32")

# n_signals_np = neutron_signals.to_numpy()
# n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
# n_signals_np = -n_signals_np + n_baselines
# print(n_signals_np.max())
# neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
neutron_signals = correct_raw_signals(neutron_signals)

# g_signals_np = gamma_signals.to_numpy()
# g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
# g_signals_np = -g_signals_np + g_baselines
# print(g_signals_np.max())
# gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)
gamma_signals = correct_raw_signals(gamma_signals)

exp_data["neutron_signals"] = neutron_signals
exp_data["gamma_signals"] = gamma_signals

In [ ]:
min_height = 13000
max_height = 14500

exp_data = experiment_neutron_data[exp_id]
neutron_signals = exp_data["neutron_signals"]
gamma_signals = exp_data["gamma_signals"]

selected_neutron = None
selected_gamma = None

bad_neutrons = [64413, 67314, 125305, 127752]
bad_gamma = []

for neutron_id, neutron_signal in neutron_signals.iterrows():
    # get clean neutron pulse (no secondary peak)
    if selected_neutron is not None:
        break
    if neutron_id in bad_neutrons:
        continue
    n_height = neutron_signal.max()
    if n_height < min_height or n_height > max_height:
        continue
    # peaks, peak_data = find_peaks(neutron_signal, height=200, prominence=50)
    # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
    # if len(filtered_peaks) == 0:
    else:
        # get matching clean gamma pulse
        for gamma_id, gamma_signal in gamma_signals.iterrows():
            if selected_gamma is not None:
                break
            if gamma_id in bad_gamma:
                continue
            g_height = gamma_signal.max()
            if abs(g_height - n_height) > (0.01 * n_height):
                continue
            # peaks, peak_data = find_peaks(gamma_signal, height=200, prominence=50)
            # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
            # if len(filtered_peaks) > 0:
            #     continue
            print(f"Neutron ID = {neutron_id}, height = {n_height}")
            print(f"Gamma ID = {gamma_id}, height = {g_height}")
            selected_neutron = neutron_id, neutron_signal
            selected_gamma = gamma_id, gamma_signal

if selected_neutron is None or selected_gamma is None:
    raise Exception("No pulse found")
exp_data["selected_neutron"] = selected_neutron
exp_data["selected_gamma"] = selected_gamma

### Figure 3e Processing

In [ ]:
exp_id = "TB-26"

In [ ]:
# Generate histogram

adc_width = 20

exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
Z, xe, ye = get_psd_adc_histogram(
    psd_report,
    adc_width=adc_width
)
adc_histogram = {
    "histogram": Z,
    "x_edges": xe,
    "y_edges": ye,
}
exp_data["adc_histogram"] = adc_histogram
# exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
# exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
# exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
def plot_figure_3a(ax: mpl.axes.Axes):
    figa_data = experiment_neutron_data[long_window_dataset_name]
    samples = figa_data["samples"]
    x_data = np.arange(0, samples.shape[1] - 5000) * 2 / 1000
    for i, row in enumerate(samples):
        # if i == 0:
        #     print(row[5000:].shape)
        ax.plot(x_data, row[5000:], linewidth=3, color="black", alpha=0.5)
    ax.set_xlim(-1, 31)
    ax.grid(alpha=0.5)
    ax.set_xlabel(r"Time ($\mu$s)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)

In [ ]:
def plot_figure_3b(ax: mpl.axes.Axes):
    exp_data = experiment_neutron_data["test_noise_floor_26.04.24"]
    signals_df = exp_data["signals_df"]
    histo_results = exp_data["histo_results"]
    e_bins = histo_results["e_bins"]
    collapsed_histo_energy = histo_results["collapsed_histo_energy"]

    # e_conversion_value = 17/100
    # threshold = 250
    
    e_mids = (e_bins[1:] + e_bins[:-1]) / 2
    # below_threshold = e_mids < threshold
    # above_threshold = e_mids >= threshold
    
    divider = make_axes_locatable(ax)
    ax_h = divider.append_axes("right", 3, pad=0, sharey=ax)

    pulse_x = signals_df.columns.map(lambda x: int(x) * 2)
    pulse_x_subset = pulse_x[:40]
    signals_np = signals_df.to_numpy()
    for pulse_y_subset in signals_np[:, :40]:
        ax.plot(pulse_x_subset, pulse_y_subset, color=bg_blue, linewidth=3, alpha=0.2)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlim(0, 80)

    ax_h.fill_betweenx(e_mids, 0, collapsed_histo_energy, color=bg_blue)
    
    ax_h.set_xscale("log")
    ax_h.tick_params(labelsize=fontsize)
    ax_h.xaxis.set_major_locator(mpl.ticker.LogLocator(numticks=5))
    ax_h.set_xlabel("Counts", fontsize=fontsize)
    ax_h.yaxis.set_tick_params(labelleft=False)

In [ ]:
def plot_figure_3c(ax: mpl.axes.Axes):
    trigger_time = t_t = 144
    pulse_time = t_omega = 400
    max_adc = 16367

    exp_data = experiment_neutron_data["TB-26"]
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values
    neutron_y = np.pad(neutron_y, 50, mode="edge")
    neutron_x = np.arange(-50, len(neutron_y) - 50) * 2

    divider = make_axes_locatable(ax)
    axes_height = 1.4
    ax2 = divider.append_axes("bottom", axes_height, pad=1.2, sharex=ax)
    
    ax.plot(neutron_x, neutron_y, label="Neutron", linewidth=3)
    n_boxes = 3
    margin = 0.05
    box_height = (1-(n_boxes-1)*margin)/n_boxes
    textbox("Pre-trigger", 0, 2*(box_height+margin), t_t, box_height, bg_grey, "white", ax2)

    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, 1),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linewidth=2,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)
    ax.text(t_t+2, trigger_line_height - 275, "CFD Trigger", ha="left", va="center", fontsize=fontsize-2)
    ax.text(pulse_time / 2, max_adc, "Acquisition Window", ha="center", va="bottom", fontsize=fontsize-2)
    box_width_offset = 2
    box_height_offset = 50
    box = mpl.patches.Rectangle(
        (box_width_offset, box_height_offset),
        pulse_time-2*box_width_offset,
        max_adc-2*box_height_offset,
        ec="black",
        fc="#f5f5f5",
        # alpha=0.1,
        zorder=0,
        lw=3
    )
    ax.add_patch(box)

    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_xlabel("\nTime (ns)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(axis="y", labelleft=False)
    ax.set_ylim(0, 17000)
    ax.set_xlim(-100, pulse_time+100)
    ax.xaxis.set_major_formatter(
        lambda x, _: str(int(x)) if x >= 0 and x <= 400 else ""
    )
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    ax.spines["bottom"].set_position("zero")
    ax.spines["left"].set_position("zero")
    # ax.spines["left"].set_linewidth(3)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
def plot_figure_3d(ax: mpl.axes.Axes):
    trigger_time = t_t = 144
    pulse_time = t_omega = 400
    pre_gate = 50
    short_gate = 22
    gate_width = 250
    
    t_1 = t_t - pre_gate
    t_2 = t_1 + short_gate
    t_3 = t_1 + gate_width
    
    exp_data = experiment_neutron_data["TB-26"]
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values
    gamma_y = selected_gamma.values
    neutron_x = np.arange(0, len(selected_neutron)) * 2
    gamma_x = np.arange(0, len(selected_gamma)) * 2

    divider = make_axes_locatable(ax)
    axes_height = 1.4
    ax2 = divider.append_axes("bottom", axes_height, pad=1.2, sharex=ax)
    
    ax.plot(neutron_x, neutron_y, label="Neutron", linewidth=3)
    ax.plot(gamma_x, gamma_y, label="Gamma", linewidth=3)
    ax.fill_between(
        neutron_x, neutron_y, gamma_y, where=(100 <= neutron_x), color=bg_blue, interpolate=True
    )

    n_boxes = 3
    margin = 0.05
    box_height = (1-(n_boxes-1)*margin)/n_boxes
    textbox("Short gate (head integral)", t_1, 0, short_gate, box_height, bg_grey, "black", ax2, ha="left", text_x_offset=1.1)
    textbox("Gate (total integral)", t_1, 1*(box_height+margin), gate_width, box_height, bg_grey, "white", ax2)
    textbox("Pre-gate", t_1, 2*(box_height+margin), pre_gate, box_height, bg_grey, "white", ax2)

    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, 2*(box_height+margin)),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linewidth=2,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)
    ax.text(t_t+2, trigger_line_height - 275, "CFD Trigger", ha="left", va="center", fontsize=fontsize-2)

    ax.set_ylim(0, 17000)
    ax.set_xlim(0, pulse_time)
    
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    # ax.yaxis.set_major_locator(mpl.ticker.LinearLocator())
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())

    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([t_1, t_2, t_3], labels=["\n$t_1$", "\n$t_2$", "\n$t_3$"])
    sec.tick_params('x', labelsize=fontsize, length=0, pad=10)
    sec.set_xlabel("Time (ns)", fontsize=fontsize)

    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
def plot_figure_3e(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    fontsize = 28
    histo_res = 128
    contour_res = 100
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    exp_data = experiment_neutron_data["TB-26"]
    histo_dict = exp_data["adc_histogram"]
    xe = histo_dict["x_edges"]
    ye = histo_dict["y_edges"]
    dz = histo_dict["histogram"]
    
    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin
    
    height_mask = dz > 20
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]

    # min_dz = np.min(dz)
    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             # color=color_alphas
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.2,
             ec="black"
            )

    bbox_params = {
        # "boxstyle": "round, pad=0.003, rounding_size=0.1",
        "fc": "white",
        "lw": 1,
        "fill": True,
        "alpha": 0.9
    }
    text_params = {
        "fontsize": fontsize-2,
        "bbox": bbox_params,
        "zorder": 5,
        "va": "center"
    }
    
    gamma_text = ["Gamma", "channel"]
    neutron_text = ["Neutron", "channel"]
    # scaling = (400, 0.025)
    scaling = (0.03, 280)
    gamma_position = (4400, 0.135)
    neutron_position = (4400, 0.35)
    # mutation_aspect = 3500 / 0.5
    # mutation_aspect = None
    linespacing = 1
    padding = 0.003
    rounding_size = 0.1
    font_properties = mpl.font_manager.FontProperties(weight="bold")

    for text_list, position in [
        (gamma_text, gamma_position),
        (neutron_text, neutron_position)
    ]:
        text_path = make_text_path(
            text_list,
            linespacing,
            # font_properties=font_properties
        )
        # bbox = make_bounding_box_for_text_path(
        #     text_path,
        #     padding,
        #     rounding_size,
        #     1,
        #     # mutation_aspect=mutation_aspect,
        #     **bbox_params
        # )
        transform = mpl.transforms.Affine2D()
        transform = transform.scale(*scaling)
        pos_from = get_bbox_center(text_path.get_extents(transform))
        x_translate, y_translate = get_translation_to(pos_from, position)
        transform = transform.translate(x_translate, y_translate)
        x_center, y_center = get_bbox_center(text_path.get_extents(transform))
        transform = transform.rotate_deg_around(x_center, y_center, 90)
        text_path = transform.transform_path(text_path)
        text_patch = convert_text_path_to_patch(text_path, 1)
        # patch_to_3d_plot_wall(bbox, ax, "z")
        patch_to_3d_plot_wall(text_patch, ax, "z")

    arrow_width = 0.04
    arrow_head_width = 0.07
    arrow_head_length = 400
    arrow_base_e = 3950
    arrow_len_psd = 0
    arrow_kwargs = {
        "width": arrow_width,
        "head_width": arrow_head_width,
        "head_length": arrow_head_length,
        "length_includes_head": True,
        "ec": "black",
        "fc": "none",
        "lw": 4
    }
    arrow_g = mpl.patches.FancyArrow(
        arrow_base_e, 0.135, -800, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_g, ax, "z")
    arrow_n = mpl.patches.FancyArrow(
        arrow_base_e, 0.35, -2550, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_n, ax, "z")

    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
    ax.set_xlim(-100, 5000)
    ax.set_ylim(-0.01, 0.51)
    ax.set_zlim(0, 4000)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=4)
    # ax.tick_params(axis="z", pad=-2)
    # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
    #     axis3d.labelpad = 40
    ax.xaxis.labelpad = 48
    ax.yaxis.labelpad = 40
    ax.zaxis.labelpad = 44
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
dl_folder = Path.home() / "Downloads"
mosaic = """
AAABB
CCDDD
.EEE.
"""
fig, ax_dict = plt.subplot_mosaic(
    mosaic,
    figsize=(17, 22),
    width_ratios=[17, 17, 8, 9, 17],
    # subplot_kw={
    # },
    # per_subplot_kw={"E": {"projection": "3d", "computed_zorder": False}},
    dpi=600,
    layout="constrained"
)
plot_figure_3a(ax_dict["A"])
plot_figure_3b(ax_dict["B"])
plot_figure_3c(ax_dict["C"])
plot_figure_3d(ax_dict["D"])
# plot_figure_3e(ax_dict["E"])
# for ax in ax_dict.values():
#     bbox = ax.get_tightbbox(fig.canvas.get_renderer())
#     x0, y0, width, height = bbox.transformed(fig.transFigure.inverted()).bounds
#     xpad = 0.01 * width
#     ypad = 0.01 * height
#     fig.add_artist(
#         plt.Rectangle(
#             (x0-xpad, y0-ypad),
#             width+2*xpad,
#             height+2*ypad,
#             ec="red",
#             lw=3,
#             fill=False
#         )
#     )
# plt.tight_layout(pad=1.01)
# fig.savefig(
#     dl_folder / "fig3.png",
#     dpi=fig.dpi,
#     bbox_inches="tight"
# )

In [ ]:
input("Processing done, hit Enter to finish")
stop()

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=(6.5, 7.3),
    # dpi=600,
    layout="constrained"
)
plot_figure_3b(ax)
# fig.savefig(dl_folder / "fig_3b.png", dpi=fig.dpi, bbox_inches="tight")